# einops-repeat — ex7: 2D positional encoding — tile a 1D PE across a spatial grid

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-repeat`. Running the final beacon cell reports progress against the `Einops: Repeat` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Repeat` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-repeat`** (exercise 7). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-repeat"
DD_SUBTOPIC = "Einops: Repeat"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.repeat — quick refresher

`repeat(tensor, pattern, **axes_lengths)` introduces new axes or stretches existing ones:
1. **New axis** — `'h w -> b h w'` with `b=4` broadcasts across a new leading dim.
2. **Stretch (nearest-neighbor)** — `'h w -> (h r) w'` with `r=2` makes each row appear twice in a contiguous block (rows 0,0,1,1,2,2,...).
3. **Tile** — `'h w -> h (r w)'` with `r=2` concatenates two full copies side-by-side (cols 0..w-1, then 0..w-1 again).

Stretch vs tile: in the composite `(a b)` the axis written **first varies slower**. `(h r)` puts source row 0 at output rows `0..r-1`; `(r h)` puts source row 0 at output rows `0, h, 2h, ...`. The new exercises lean on this distinction repeatedly.

### Exercise 7 — 2D positional encoding — tile a 1D PE across a spatial grid

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Build a 2D positional encoding by repeating a 1D row encoding across the column axis (and a 1D column encoding across the row axis), then sum them.
> Keywords: positional-encoding, tile, broadcast, visualization
> ```

**KCs targeted:** `repeat-add-axis`, `repeat-broadcast-vs-tile`

Implement `ex7_make_2d_pe(pe_row, pe_col)`.

Input: `pe_row` is a `(H, D)` 1D positional encoding indexed by row, `pe_col` is a `(W, D)` 1D positional encoding indexed by column. Output: `(H, W, D)` — the standard "separable" 2D PE you sum onto a flattened image patch grid.

Construct the output by **repeating** `pe_row` across the new `W` axis and `pe_col` across the new `H` axis, then summing the two `(H, W, D)` tensors. Use one `einops.repeat` per operand — no `unsqueeze`, `expand`, or `broadcast_to`.

The test cell visualizes the resulting `(H, W)` map at one feature channel (`d=0`) as a heatmap. You should be able to see vertical stripes from `pe_col` and horizontal stripes from `pe_row` superposed.

In [ ]:
def ex7_make_2d_pe(pe_row: Tensor, pe_col: Tensor) -> Tensor:
    """Combine a (H, D) row PE and (W, D) col PE into a (H, W, D) 2D PE."""
    raise NotImplementedError()


def _test_ex7():
    H, W, D = 8, 12, 16
    # Simple sinusoidal-ish 1D PEs (we just need something visually distinct).
    rows = t.linspace(0, 3.14159, H).unsqueeze(1) * (t.arange(D).float() + 1)
    cols = t.linspace(0, 3.14159, W).unsqueeze(1) * (t.arange(D).float() + 1)
    pe_row = t.sin(rows)         # (H, D)
    pe_col = t.cos(cols)         # (W, D)

    out = ex7_make_2d_pe(pe_row, pe_col)

    assert out.shape == (H, W, D), f'expected ({H},{W},{D}), got {out.shape}'
    # Each (h, w, :) should equal pe_row[h] + pe_col[w].
    for h in [0, H // 2, H - 1]:
        for w in [0, W // 2, W - 1]:
            expected = pe_row[h] + pe_col[w]
            assert t.allclose(out[h, w], expected, atol=1e-6), f'mismatch at ({h},{w})'

    # Visualize one feature channel as a 2D heatmap.
    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    axes[0].imshow(repeat(pe_row[:, 0], 'h -> h w', w=W).numpy(), cmap='RdBu_r')
    axes[0].set_title('pe_row[:,0] broadcast')
    axes[1].imshow(repeat(pe_col[:, 0], 'w -> h w', h=H).numpy(), cmap='RdBu_r')
    axes[1].set_title('pe_col[:,0] broadcast')
    axes[2].imshow(out[..., 0].numpy(), cmap='RdBu_r')
    axes[2].set_title('summed 2D PE (channel 0)')
    for ax in axes:
        ax.set_xlabel('col w'); ax.set_ylabel('row h')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex7')
    print("ex7 ✓")

_test_ex7()

<details><summary>Solution</summary>

```python
def ex7_make_2d_pe(pe_row: Tensor, pe_col: Tensor) -> Tensor:
    row_grid = repeat(pe_row, 'h d -> h w d', w=pe_col.shape[0])
    col_grid = repeat(pe_col, 'w d -> h w d', h=pe_row.shape[0])
    return row_grid + col_grid
```

**Why two repeats then a sum.** `pe_row` has no `w` axis; `pe_col` has no `h` axis. You can't add them directly because their shapes don't broadcast (`(H, D)` vs `(W, D)`). Repeating each one into the full `(H, W, D)` grid is the explicit, pattern-driven way to align them — torch's implicit broadcasting would require `unsqueeze`s in the right slots, which is exactly the bookkeeping einops eliminates.

**The three-panel plot** shows the two stripe patterns and their sum, which is the standard separable 2D positional encoding used in vision transformers (ViT, MAE).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex7',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()